# 🧮 임베딩 & FAISS 인덱스 관리자 만들기

## 🎯 실습 목표
- 텍스트를 임베딩 벡터로 변환하는 **함수**들을 직접 만들어보기
- FAISS 벡터 인덱스를 생성하고 관리하는 방법 익히기
- 각 함수를 테스트하면서 동작 확인하기
- 마지막에 모든 함수를 하나의 **클래스**로 합치기

---

## 목적
RAG 시스템에서 문서 검색을 가능하게 하기 위해 텍스트를 벡터로 변환하고 인덱싱하는 작업입니다. AI 챗봇이 사용자의 질문과 가장 관련성 높은 문서를 빠르게 찾으려면, 텍스트를 수학적으로 비교 가능한 벡터 형태로 변환해야 합니다. 이 임베딩 벡터들을 FAISS 인덱스에 저장하면 수천, 수만 개의 문서 중에서도 밀리초 단위로 유사한 문서를 검색할 수 있습니다.

## 기능
OpenAI의 text-embedding-3-large 모델을 사용해 텍스트를 3072차원의 벡터로 변환합니다. 임베딩 결과는 캐시에 저장되어 동일한 텍스트에 대한 중복 API 호출을 방지합니다. 변환된 벡터들은 FAISS IndexFlatL2 인덱스에 저장되어 L2 거리 기반의 정확한 최근접 이웃 검색이 가능합니다. 청크 단위로 배치 처리하여 대량의 문서도 효율적으로 임베딩합니다.

## 결과물
FAISS 인덱스 파일(faiss_index.bin)과 메타데이터 파일(metadata.json)이 생성됩니다. 인덱스 파일에는 모든 청크의 임베딩 벡터가 저장되고, 메타데이터 파일에는 각 벡터의 원본 텍스트와 출처 정보가 저장됩니다. 또한 임베딩 캐시 파일(embeddings_*.pkl)이 생성되어 이후 동일한 텍스트에 대한 재임베딩 비용을 절약합니다. 이 파일들은 다음 단계인 검색 엔진에서 사용됩니다.

---

## 🔧 Step 1: 환경 설정

필요한 라이브러리를 임포트하고 API 키를 설정합니다.

In [ ]:
# 필요한 라이브러리 설치
import sys
!{sys.executable} -m pip install openai faiss-cpu python-dotenv --break-system-packages -q

In [ ]:
# 필요한 라이브러리 임포트
import os
import json
import pickle
import hashlib
from typing import List, Dict, Optional, Tuple
import numpy as np
from openai import OpenAI
import faiss
from dotenv import load_dotenv

print("✅ 라이브러리 임포트 완료")

In [ ]:
# 환경 변수 로드 및 OpenAI 클라이언트 설정
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY:
    print(f"API Key: {OPENAI_API_KEY[:8]}...")
else:
    print("API Key not found. .env 파일에 OPENAI_API_KEY를 설정해주세요.")

In [ ]:
# 기본 설정값
MODEL = "text-embedding-3-large"  # 임베딩 모델
DIMENSION = 3072                   # 임베딩 차원

# 경로 설정 (실습 환경에 맞게 수정)
DATA_DIR = "data"
CHUNKS_PATH = os.path.join(DATA_DIR, "chunks", "construction_law_chunks.json")
CACHE_DIR = os.path.join(DATA_DIR, "cache")
OUTPUT_DIR = os.path.join(DATA_DIR, "vector_store", "construction_law")

print(f"📁 청크 파일: {CHUNKS_PATH}")
print(f"📁 캐시 디렉토리: {CACHE_DIR}")
print(f"📁 출력 디렉토리: {OUTPUT_DIR}")

---

## 🔧 Step 2: 텍스트 해시 계산

### 왜 해시가 필요할까요?
- 긴 텍스트를 캐시 키로 직접 사용하면 메모리 낭비
- 해시는 텍스트의 고유한 "지문" 역할
- 동일한 텍스트는 항상 동일한 해시값 생성

### ToDo #1: `get_text_hash()` 함수 만들기

**목표**: 텍스트의 MD5 해시 계산

**힌트**:
- `hashlib.md5(text.encode('utf-8'))`로 해시 객체 생성
- `.hexdigest()`로 16진수 문자열 반환

In [ ]:
def get_text_hash(text: str) -> str:
    """
    텍스트의 MD5 해시를 계산합니다.
    
    Args:
        text: 해시할 텍스트
    
    Returns:
        str: 32자리 16진수 해시 문자열
    """
    # ToDo #1: 여기에 코드를 작성하세요
    # 1. hashlib.md5()로 해시 객체 생성
    # 2. text.encode('utf-8')로 바이트로 변환
    # 3. .hexdigest()로 문자열 반환
    pass

### ✅ 테스트 #1: 해시 계산

In [ ]:
# 해시 테스트
test_text = "건축법 제1조 목적"
hash_result = get_text_hash(test_text)

print(f"✅ 해시 계산 완료")
print(f"  텍스트: {test_text}")
print(f"  해시: {hash_result}")
print(f"  해시 길이: {len(hash_result)}자")

# 동일 텍스트 = 동일 해시 확인
hash_result2 = get_text_hash(test_text)
print(f"\n동일 텍스트 재해시: {hash_result == hash_result2}")

---

## 🔧 Step 3: 임베딩 캐시 관리

### 왜 캐시가 필요할까요?
- OpenAI 임베딩 API는 호출당 비용이 발생합니다
- 동일한 텍스트를 다시 임베딩하면 불필요한 비용과 시간이 소요됩니다
- 캐시를 사용하면 한 번 생성한 임베딩을 재사용할 수 있습니다

### ToDo #2: `load_embedding_cache()` 함수 만들기

**목표**: pickle 파일에서 캐시 로드

**힌트**:
- `os.path.exists()`로 파일 존재 확인
- `pickle.load()`로 딕셔너리 로드
- 파일이 없으면 빈 딕셔너리 `{}` 반환

In [ ]:
def load_embedding_cache(cache_path: str) -> Dict[str, np.ndarray]:
    """
    임베딩 캐시를 파일에서 로드합니다.
    
    Args:
        cache_path: 캐시 파일 경로
    
    Returns:
        Dict[str, np.ndarray]: {텍스트 해시: 임베딩 벡터} 딕셔너리
    """
    # ToDo #2: 여기에 코드를 작성하세요
    # 1. os.path.exists()로 파일 존재 확인
    # 2. 파일이 있으면 pickle.load()로 로드 (open은 'rb' 모드)
    # 3. 파일이 없으면 빈 딕셔너리 {} 반환
    pass

### ToDo #3: `save_embedding_cache()` 함수 만들기

**목표**: 캐시를 pickle 파일로 저장

**힌트**:
- `pickle.dump(cache, f)`로 저장
- `'wb'` 모드로 파일 열기

In [ ]:
def save_embedding_cache(cache: Dict[str, np.ndarray], cache_path: str):
    """
    임베딩 캐시를 파일에 저장합니다.
    
    Args:
        cache: 임베딩 캐시 딕셔너리
        cache_path: 저장할 파일 경로
    """
    # ToDo #3: 여기에 코드를 작성하세요
    # 1. open()으로 파일 열기 ('wb' 모드)
    # 2. pickle.dump(cache, f)로 저장
    pass

### ✅ 테스트 #2-3: 캐시 저장/로드

In [ ]:
# 캐시 테스트
os.makedirs(CACHE_DIR, exist_ok=True)
test_cache_path = os.path.join(CACHE_DIR, "test_cache.pkl")

# 테스트 데이터
test_cache = {
    "hash1": np.array([0.1, 0.2, 0.3], dtype='float32'),
    "hash2": np.array([0.4, 0.5, 0.6], dtype='float32')
}

# 저장 테스트
save_embedding_cache(test_cache, test_cache_path)
print(f"✅ 캐시 저장 완료: {test_cache_path}")

# 로드 테스트
loaded_cache = load_embedding_cache(test_cache_path)
print(f"✅ 캐시 로드 완료: {len(loaded_cache)}개 항목")

# 정리
os.remove(test_cache_path)

---

## 🔧 Step 4: 단일 텍스트 임베딩

### OpenAI 임베딩 API 사용법
```python
response = client.embeddings.create(
    input=text,
    model="text-embedding-3-large"
)
embedding = response.data[0].embedding  # 3072차원 리스트
```

In [ ]:
# 전역 캐시 (실습용)
embedding_cache = {}

def embed_text(text: str, 
               client: OpenAI,
               model: str = "text-embedding-3-large",
               dimension: int = 3072) -> np.ndarray:
    """
    텍스트를 임베딩 벡터로 변환합니다.
    
    Args:
        text: 임베딩할 텍스트
        client: OpenAI 클라이언트
        model: 임베딩 모델명
        dimension: 임베딩 차원
    
    Returns:
        np.ndarray: 임베딩 벡터 (float32)
    """
    global embedding_cache
    
    # 캐시 확인
    text_hash = get_text_hash(text)
    if text_hash in embedding_cache:
        return embedding_cache[text_hash]
    
    # OpenAI API 호출
    try:
        response = client.embeddings.create(
            input=text,
            model=model
        )
        
        embedding = np.array(response.data[0].embedding, dtype='float32')
        
        # 캐시에 저장
        embedding_cache[text_hash] = embedding
        
        return embedding
        
    except Exception as e:
        print(f"⚠ 임베딩 생성 실패: {e}")
        return np.zeros(dimension, dtype='float32')

### ✅ 테스트: 단일 텍스트 임베딩

In [ ]:
# 단일 임베딩 테스트
test_text = "건축법 제1조 이 법은 건축물의 대지·구조·설비 기준 및 용도 등을 정하여 건축물의 안전·기능·환경 및 미관을 향상시킴으로써 공공복리의 증진에 이바지하는 것을 목적으로 한다."

embedding = embed_text(test_text, client)

print(f"✅ 임베딩 생성 완료")
print(f"  차원: {len(embedding)}")
print(f"  타입: {embedding.dtype}")
print(f"  처음 5개 값: {embedding[:5]}")

# 캐시 확인
print(f"\n캐시 항목 수: {len(embedding_cache)}")

In [ ]:
# 캐시 히트 테스트 (동일 텍스트 재요청)
import time

start = time.time()
embedding2 = embed_text(test_text, client)  # 캐시에서 가져옴
elapsed = time.time() - start

print(f"✅ 캐시 히트!")
print(f"  소요 시간: {elapsed:.4f}초 (거의 0초)")
print(f"  동일 결과: {np.array_equal(embedding, embedding2)}")

---

## 🔧 Step 5: 배치 임베딩

### 왜 배치 처리가 필요할까요?
- OpenAI API는 여러 텍스트를 한 번에 임베딩 가능
- 개별 호출보다 훨씬 효율적 (네트워크 오버헤드 감소)
- 대량의 문서를 빠르게 처리

In [ ]:
def embed_chunks(chunks: List[Dict], 
                 client: OpenAI,
                 model: str = "text-embedding-3-large",
                 dimension: int = 3072,
                 batch_size: int = 100) -> Tuple[List[np.ndarray], List[str]]:
    """
    여러 청크를 배치로 임베딩합니다.
    
    Args:
        chunks: 청크 리스트 (각 청크는 'content', 'chunk_id' 포함)
        client: OpenAI 클라이언트
        model: 임베딩 모델명
        dimension: 임베딩 차원
        batch_size: 배치 크기
    
    Returns:
        Tuple[List[np.ndarray], List[str]]: (임베딩 리스트, 청크 ID 리스트)
    """
    global embedding_cache
    
    embeddings = []
    chunk_ids = []
    
    print(f"\n🧮 임베딩 생성 시작...")
    print(f"  - 총 청크 수: {len(chunks)}")
    print(f"  - 배치 크기: {batch_size}")
    
    cache_hits = 0
    cache_misses = 0
    
    for i in range(0, len(chunks), batch_size):
        batch = chunks[i:i+batch_size]
        batch_texts = [chunk['content'] for chunk in batch]
        batch_chunk_ids = [chunk['chunk_id'] for chunk in batch]
        
        print(f"\n  배치 {i//batch_size + 1}/{(len(chunks)-1)//batch_size + 1}")
        
        # 배치 내에서 캐시 확인
        batch_embeddings = []
        texts_to_embed = []
        text_indices = []
        
        for j, text in enumerate(batch_texts):
            text_hash = get_text_hash(text)
            if text_hash in embedding_cache:
                batch_embeddings.append(embedding_cache[text_hash])
                cache_hits += 1
            else:
                batch_embeddings.append(None)
                texts_to_embed.append(text)
                text_indices.append(j)
                cache_misses += 1
        
        # 캐시에 없는 것만 API 호출
        if texts_to_embed:
            try:
                response = client.embeddings.create(
                    input=texts_to_embed,
                    model=model
                )
                
                for idx, data in enumerate(response.data):
                    embedding = np.array(data.embedding, dtype='float32')
                    original_idx = text_indices[idx]
                    batch_embeddings[original_idx] = embedding
                    
                    # 캐시에 저장
                    text_hash = get_text_hash(texts_to_embed[idx])
                    embedding_cache[text_hash] = embedding
                
                print(f"    ✓ {len(texts_to_embed)}개 새로 생성")
                
            except Exception as e:
                print(f"    ✗ 배치 실패: {e}")
                for idx in text_indices:
                    if batch_embeddings[idx] is None:
                        batch_embeddings[idx] = np.zeros(dimension, dtype='float32')
        
        embeddings.extend(batch_embeddings)
        chunk_ids.extend(batch_chunk_ids)
        
        progress = min((i + batch_size) / len(chunks) * 100, 100)
        print(f"    진행률: {progress:.1f}%")
    
    print(f"\n✅ 임베딩 생성 완료!")
    print(f"  - 캐시 히트: {cache_hits}개")
    print(f"  - 새로 생성: {cache_misses}개")
    
    return embeddings, chunk_ids

### ✅ 테스트: 배치 임베딩

In [ ]:
# 테스트용 청크 데이터
test_chunks = [
    {"chunk_id": "chunk_001", "content": "건축법 제1조 목적"},
    {"chunk_id": "chunk_002", "content": "건축법 제2조 정의"},
    {"chunk_id": "chunk_003", "content": "건축법 제3조 적용제외"}
]

embeddings, chunk_ids = embed_chunks(test_chunks, client, batch_size=2)

print(f"\n결과 확인:")
print(f"  임베딩 수: {len(embeddings)}")
print(f"  청크 ID 수: {len(chunk_ids)}")
print(f"  첫 번째 임베딩 차원: {len(embeddings[0])}")

---

## 🔧 Step 6: FAISS 인덱스 생성

### FAISS란?
- Facebook AI Research에서 개발한 벡터 검색 라이브러리
- 수백만 개의 벡터에서 밀리초 단위로 유사 벡터 검색
- GPU 가속 지원

### IndexFlatL2
- 가장 기본적인 인덱스 타입
- L2 거리 (유클리드 거리) 기반
- 정확한 최근접 이웃 검색 (Exact search)

In [ ]:
def create_faiss_index(embeddings: List[np.ndarray], 
                       dimension: int = 3072) -> faiss.Index:
    index = faiss.IndexFlatL2(dimension)
    embeddings_array = np.array(embeddings).astype('float32')
    index.add(embeddings_array)
    return index

### ✅ 테스트 #4: FAISS 인덱스 생성

In [ ]:
# FAISS 인덱스 생성 테스트
index = create_faiss_index(embeddings, dimension=DIMENSION)

print(f"✅ FAISS 인덱스 생성 완료")
print(f"  타입: {type(index).__name__}")
print(f"  벡터 수: {index.ntotal}")
print(f"  차원: {index.d}")

---

## 🔧 Step 7: 인덱스 및 메타데이터 저장/로드

In [ ]:
def save_index(index: faiss.Index, index_path: str):
    """FAISS 인덱스를 파일에 저장합니다."""
    os.makedirs(os.path.dirname(index_path), exist_ok=True)
    faiss.write_index(index, index_path)
    print(f"✓ 인덱스 저장: {index_path}")


def load_index(index_path: str) -> Optional[faiss.Index]:
    """FAISS 인덱스를 파일에서 로드합니다."""
    if not os.path.exists(index_path):
        print(f"⚠ 인덱스 없음: {index_path}")
        return None
    
    index = faiss.read_index(index_path)
    print(f"✓ 인덱스 로드: {index_path} ({index.ntotal}개 벡터)")
    return index

In [ ]:
def save_metadata(chunks: List[Dict], chunk_ids: List[str], metadata_path: str):
    """메타데이터를 JSON 파일로 저장합니다."""
    os.makedirs(os.path.dirname(metadata_path), exist_ok=True)
    
    # chunk_id로 청크 검색을 위한 딕셔너리
    chunk_dict = {chunk['chunk_id']: chunk for chunk in chunks}
    
    metadata = []
    for i, chunk_id in enumerate(chunk_ids):
        chunk = chunk_dict.get(chunk_id, {})
        metadata.append({
            "index": i,
            "chunk_id": chunk_id,
            "content": chunk.get("content", ""),
            "metadata": chunk.get("metadata", {})
        })
    
    with open(metadata_path, 'w', encoding='utf-8') as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)
    
    print(f"✓ 메타데이터 저장: {metadata_path} ({len(metadata)}개 항목)")


def load_metadata(metadata_path: str) -> Optional[List[Dict]]:
    """메타데이터를 JSON 파일에서 로드합니다."""
    if not os.path.exists(metadata_path):
        print(f"⚠ 메타데이터 없음: {metadata_path}")
        return None
    
    with open(metadata_path, 'r', encoding='utf-8') as f:
        metadata = json.load(f)
    
    print(f"✓ 메타데이터 로드: {metadata_path} ({len(metadata)}개 항목)")
    return metadata

### ✅ 테스트: 저장/로드

In [ ]:
# 저장 테스트
os.makedirs(OUTPUT_DIR, exist_ok=True)
test_index_path = os.path.join(OUTPUT_DIR, "test_index.bin")
test_metadata_path = os.path.join(OUTPUT_DIR, "test_metadata.json")

save_index(index, test_index_path)
save_metadata(test_chunks, chunk_ids, test_metadata_path)

# 로드 테스트
loaded_index = load_index(test_index_path)
loaded_metadata = load_metadata(test_metadata_path)

print(f"\n첫 번째 메타데이터:")
print(json.dumps(loaded_metadata[0], ensure_ascii=False, indent=2))

# 정리
os.remove(test_index_path)
os.remove(test_metadata_path)

In [ ]:
"""
s4_EmbeddingManager.py
임베딩 생성과 FAISS 인덱스 관리
"""

"""
✅ FAISS 인덱스는 "벡터 창고"
┌─────────────────────────────────────────────────────────┐
│ FAISS 인덱스 = 1500개 벡터를 그냥 저장해둔 창고        │
│                                                         │
│ 📦 벡터 0:    [0.023, -0.056, 0.089, ...]              │
│ 📦 벡터 1:    [0.045, 0.012, -0.034, ...]              │
│ 📦 벡터 2:    [-0.078, 0.091, 0.056, ...]              │
│ ...                                                     │
│ 📦 벡터 1499: [0.034, -0.067, 0.045, ...]              │
│                                                         │
│ → 미리 정렬되어 있지 않음!                             │
│ → 검색할 때 거리 계산해서 정렬함                       │
└─────────────────────────────────────────────────────────┘
"""

import os
import json
import pickle
import hashlib
from typing import List, Dict, Optional, Tuple
import numpy as np
from openai import OpenAI
import faiss
from dotenv import load_dotenv


class EmbeddingManager:
    """임베딩 생성 및 FAISS 인덱스 관리 클래스"""
    
    def __init__(self, 
                 openai_api_key: str,
                 institution: str = "construction_law",
                 model: str = "text-embedding-3-large",
                 cache_dir: str = None,
                 dimension: int = 3072):
        """
        Args:
            openai_api_key: OpenAI API 키
            institution: 기관/프로젝트 이름 (캐시 파일명용)
            model: 임베딩 모델명
            cache_dir: 캐시 디렉토리 경로 (절대 경로 권장, None이면 "data/cache")
            dimension: 임베딩 차원
        """
        self.client = OpenAI(api_key=openai_api_key)
        self.model = model
        self.institution = institution
        self.dimension = dimension
        
        # 캐시 경로 자동 생성
        if cache_dir is None:
            cache_dir = "data/cache"
        
        os.makedirs(cache_dir, exist_ok=True)
        cache_path = os.path.join(cache_dir, f"embeddings_{institution}.pkl")
        
        self.cache_path = cache_path
        self.embedding_cache = self.load_embedding_cache()
        
        print(f"🎯 EmbeddingManager 초기화")
        print(f"  - 모델: {model}")
        print(f"  - 차원: {dimension}")
        print(f"  - 캐시: {len(self.embedding_cache)}개 임베딩")
        print(f"  - 캐시 경로: {cache_path}")
    
    def load_embedding_cache(self) -> Dict[str, np.ndarray]:
        """임베딩 캐시 로드"""
        if os.path.exists(self.cache_path):
            try:
                with open(self.cache_path, 'rb') as f:
                    cache = pickle.load(f)
                print(f"✓ 캐시 로드: {len(cache)}개 임베딩")
                return cache
            except Exception as e:
                print(f"⚠ 캐시 로드 실패 ({e}), 새로 시작")
                return {}
        return {}
    
    def save_embedding_cache(self):
        """임베딩 캐시 저장"""
        try:
            with open(self.cache_path, 'wb') as f:
                pickle.dump(self.embedding_cache, f)
            print(f"✓ 캐시 저장: {len(self.embedding_cache)}개 임베딩")
        except Exception as e:
            print(f"⚠ 캐시 저장 실패: {e}")
    
    def get_text_hash(self, text: str) -> str:
        """텍스트의 MD5 해시 계산 (캐시 키)"""
        return hashlib.md5(text.encode('utf-8')).hexdigest()
    
    def embed_text(self, text: str) -> np.ndarray:
        """텍스트를 임베딩 벡터로 변환"""
        # 캐시 확인
        text_hash = self.get_text_hash(text)
        if text_hash in self.embedding_cache:
            return self.embedding_cache[text_hash]
        
        # OpenAI API 호출
        try:
            response = self.client.embeddings.create(
                input=text,
                model=self.model
            )
            
            embedding = np.array(response.data[0].embedding, dtype='float32')
            
            # 캐시에 저장
            self.embedding_cache[text_hash] = embedding
            
            return embedding
            
        except Exception as e:
            print(f"⚠ 임베딩 생성 실패: {e}")
            return np.zeros(self.dimension, dtype='float32')
    
    def embed_chunks(self, chunks: List[Dict], batch_size: int = 100) -> Tuple[List[np.ndarray], List[str]]:
        """
        여러 청크를 배치로 임베딩
        
        Returns:
            (임베딩 벡터 리스트, 청크 ID 리스트)
        """
        embeddings = []
        chunk_ids = []
        
        print(f"\n🧮 임베딩 생성 시작...")
        print(f"  - 총 청크 수: {len(chunks)}")
        print(f"  - 배치 크기: {batch_size}")
        
        cache_hits = 0
        cache_misses = 0
        
        for i in range(0, len(chunks), batch_size):
            batch = chunks[i:i+batch_size]
            batch_texts = [chunk['content'] for chunk in batch]
            batch_chunk_ids = [chunk['chunk_id'] for chunk in batch]
            
            print(f"\n  배치 {i//batch_size + 1}/{(len(chunks)-1)//batch_size + 1}")
            
            # 배치 내에서 캐시 확인
            batch_embeddings = []
            texts_to_embed = []
            text_indices = []
            
            for j, text in enumerate(batch_texts):
                text_hash = self.get_text_hash(text)
                if text_hash in self.embedding_cache:
                    batch_embeddings.append(self.embedding_cache[text_hash])
                    cache_hits += 1
                else:
                    batch_embeddings.append(None)
                    texts_to_embed.append(text)
                    text_indices.append(j)
                    cache_misses += 1
            
            # 캐시에 없는 것만 API 호출
            if texts_to_embed:
                try:
                    response = self.client.embeddings.create(
                        input=texts_to_embed,
                        model=self.model
                    )
                    
                    for idx, data in enumerate(response.data):
                        embedding = np.array(data.embedding, dtype='float32')
                        original_idx = text_indices[idx]
                        batch_embeddings[original_idx] = embedding
                        
                        # 캐시에 저장
                        text_hash = self.get_text_hash(texts_to_embed[idx])
                        self.embedding_cache[text_hash] = embedding
                    
                    print(f"    ✓ {len(texts_to_embed)}개 새로 생성")
                    
                except Exception as e:
                    print(f"    ✗ 배치 실패: {e}")
                    for idx in text_indices:
                        if batch_embeddings[idx] is None:
                            batch_embeddings[idx] = np.zeros(self.dimension, dtype='float32')
            
            embeddings.extend(batch_embeddings)
            chunk_ids.extend(batch_chunk_ids)
            
            progress = min((i + batch_size) / len(chunks) * 100, 100)
            print(f"    진행률: {progress:.1f}%")
        
        print(f"\n✅ 임베딩 생성 완료!")
        print(f"  - 캐시 히트: {cache_hits}개")
        print(f"  - 새로 생성: {cache_misses}개")
        
        if cache_misses > 0:
            self.save_embedding_cache()
        
        return embeddings, chunk_ids
    
    def create_faiss_index(self, embeddings: List[np.ndarray]) -> faiss.Index:
        """FAISS 인덱스 생성"""
        print(f"\n🔧 FAISS 인덱스 생성 중...")
        
        # Flat 인덱스 (정확한 최근접 이웃 검색)
        index = faiss.IndexFlatL2(self.dimension)
        
        # numpy 배열로 변환
        embeddings_array = np.array(embeddings).astype('float32')
        
        # 인덱스에 추가
        index.add(embeddings_array)
        
        print(f"✓ FAISS 인덱스 생성 완료")
        print(f"  - 타입: Flat (L2)")
        print(f"  - 벡터 수: {index.ntotal}")
        
        return index
    
    def save_index(self, index: faiss.Index, index_path: str):
        """FAISS 인덱스 저장"""
        os.makedirs(os.path.dirname(index_path), exist_ok=True)
        
        try:
            faiss.write_index(index, index_path)
            print(f"✓ 인덱스 저장: {index_path}")
        except Exception as e:
            print(f"✗ 인덱스 저장 실패: {e}")
    
    def load_index(self, index_path: str) -> Optional[faiss.Index]:
        """FAISS 인덱스 로드"""
        if not os.path.exists(index_path):
            print(f"⚠ 인덱스 없음: {index_path}")
            return None
        
        try:
            index = faiss.read_index(index_path)
            print(f"✓ 인덱스 로드: {index_path}")
            print(f"  - 벡터 수: {index.ntotal}")
            return index
        except Exception as e:
            print(f"✗ 인덱스 로드 실패: {e}")
            return None
    
    def save_metadata(self, chunks: List[Dict], chunk_ids: List[str], metadata_path: str):
        """메타데이터 저장"""
        os.makedirs(os.path.dirname(metadata_path), exist_ok=True)
        
        chunk_dict = {chunk['chunk_id']: chunk for chunk in chunks}
        
        metadata = []
        for i, chunk_id in enumerate(chunk_ids):
            chunk = chunk_dict.get(chunk_id, {})
            metadata.append({
                "index": i,
                "chunk_id": chunk_id,
                "content": chunk.get("content", ""),
                "metadata": chunk.get("metadata", {})
            })
        
        try:
            with open(metadata_path, 'w', encoding='utf-8') as f:
                json.dump(metadata, f, ensure_ascii=False, indent=2)
            print(f"✓ 메타데이터 저장: {metadata_path}")
            print(f"  - 항목 수: {len(metadata)}")
        except Exception as e:
            print(f"✗ 메타데이터 저장 실패: {e}")
    
    def load_metadata(self, metadata_path: str) -> Optional[List[Dict]]:
        """메타데이터 로드"""
        if not os.path.exists(metadata_path):
            print(f"⚠ 메타데이터 없음: {metadata_path}")
            return None
        
        try:
            with open(metadata_path, 'r', encoding='utf-8') as f:
                metadata = json.load(f)
            print(f"✓ 메타데이터 로드: {metadata_path}")
            print(f"  - 항목 수: {len(metadata)}")
            return metadata
        except Exception as e:
            print(f"✗ 메타데이터 로드 실패: {e}")
            return None
    
    def build_index_from_chunks(self, chunks_path: str, 
                                output_dir: str = None) -> Tuple[faiss.Index, List[Dict]]:
        """
        청크 파일에서 인덱스 구축 (전체 파이프라인)
        
        Returns:
            (FAISS 인덱스, 메타데이터 리스트)
        """
        if output_dir is None:
            output_dir = f"data/vector_store/{self.institution}"
        
        print("\n" + "="*80)
        print("🚀 FAISS 인덱스 구축 시작")
        print("="*80)
        
        # 1. 청크 로드
        print("\n[1] 청크 로드")
        with open(chunks_path, 'r', encoding='utf-8') as f:
            chunks = json.load(f)
        print(f"✓ {len(chunks)}개 청크")
        
        # 2. 임베딩 생성
        print("\n[2] 임베딩 생성")
        embeddings, chunk_ids = self.embed_chunks(chunks, batch_size=100)
        
        # 3. FAISS 인덱스 생성
        print("\n[3] FAISS 인덱스 생성")
        index = self.create_faiss_index(embeddings)
        
        # 4. 저장
        print("\n[4] 저장")
        index_path = os.path.join(output_dir, "faiss_index.bin")
        self.save_index(index, index_path)
        
        metadata_path = os.path.join(output_dir, "metadata.json")
        self.save_metadata(chunks, chunk_ids, metadata_path)
        
        metadata = self.load_metadata(metadata_path)
        
        print("\n" + "="*80)
        print("✅ 인덱스 구축 완료!")
        print("="*80)
        print(f"📁 출력: {output_dir}")
        print("="*80 + "\n")
        
        return index, metadata


def main():
    """
    EmbeddingManager 실행 메인 함수
    
    입력: data/chunks/construction_law_chunks.json
    출력: data/vector_store/construction_law/
           - faiss_index.bin
           - metadata.json
    캐시: data/cache/embeddings_construction_law.pkl
    """
    print("="*80)
    print("🧮 임베딩 및 FAISS 인덱스 구축")
    print("="*80)
    
    # 환경 변수 로드
    load_dotenv()
    openai_api_key = os.getenv("OPENAI_API_KEY")
    
    if not openai_api_key:
        print("\n✗ 오류: OPENAI_API_KEY 환경 변수가 설정되지 않았습니다.")
        print("  .env 파일에 다음과 같이 설정해주세요:")
        print("  OPENAI_API_KEY=sk-your-api-key-here")
        return
    
    # 현재 파일 위치 기준으로 프로젝트 루트 찾기
    current_dir = os.path.dirname(os.path.abspath(__file__))  # src/
    project_root = os.path.dirname(current_dir)  # CNTCHATBOT_PJT2/
    
    # 경로 설정 (모두 절대 경로)
    CHUNKS_PATH = os.path.join(project_root, "data", "chunks", "construction_law_chunks.json")
    OUTPUT_DIR = os.path.join(project_root, "data", "vector_store", "construction_law")
    CACHE_DIR = os.path.join(project_root, "data", "cache")
 
    print(f"\n프로젝트 루트: {project_root}")
    print(f"입력 파일: {CHUNKS_PATH}")
    print(f"출력 디렉토리: {OUTPUT_DIR}")
    print(f"캐시 디렉토리: {CACHE_DIR}")
    
    # 입력 파일 존재 확인
    if not os.path.exists(CHUNKS_PATH):
        print(f"\n✗ 입력 파일이 없습니다: {CHUNKS_PATH}")
        print("먼저 s3_LegalChunking.py를 실행해주세요.")
        return
    
    # 이미 인덱스가 있으면 건너뛰기 옵션
    index_path = os.path.join(OUTPUT_DIR, "faiss_index.bin")
    metadata_path = os.path.join(OUTPUT_DIR, "metadata.json")
    
    if os.path.exists(index_path) and os.path.exists(metadata_path):
        response = input(f"\n⚠ 인덱스가 이미 존재합니다: {OUTPUT_DIR}\n덮어쓰시겠습니까? (y/n): ")
        if response.lower() != 'y':
            print("취소되었습니다.")
            return
    
    # EmbeddingManager 실행
    try:
        embedding_manager = EmbeddingManager(
            openai_api_key=openai_api_key,
            institution="construction_law",
            model="text-embedding-3-large",
            cache_dir=CACHE_DIR
        )
        
        index, metadata = embedding_manager.build_index_from_chunks(
            chunks_path=CHUNKS_PATH,
            output_dir=OUTPUT_DIR
        )
        
        print("="*80)
        print("✅ FAISS 인덱스 구축 완료!")
        print("="*80)
        print(f"\n생성된 파일:")
        print(f"  - {os.path.join(OUTPUT_DIR, 'faiss_index.bin')}")
        print(f"  - {os.path.join(OUTPUT_DIR, 'metadata.json')}")
        print(f"  - {os.path.join(CACHE_DIR, 'embeddings_construction_law.pkl')}")
        print("="*80 + "\n")
        
    except Exception as e:
        print(f"\n✗ 오류 발생: {e}")
        import traceback
        traceback.print_exc()


if __name__ == "__main__":
    main()

---

## 🔧 Step 8: 간단한 검색 테스트

FAISS 인덱스가 잘 작동하는지 확인해봅시다.

In [ ]:
def search_similar(query: str, 
                   index: faiss.Index,
                   metadata: List[Dict],
                   client: OpenAI,
                   top_k: int = 3) -> List[Dict]:
    """
    쿼리와 유사한 청크를 검색합니다.
    
    Args:
        query: 검색 쿼리
        index: FAISS 인덱스
        metadata: 메타데이터 리스트
        client: OpenAI 클라이언트
        top_k: 반환할 결과 수
    
    Returns:
        List[Dict]: 검색 결과 (거리, 내용 포함)
    """
    # 쿼리 임베딩
    query_embedding = embed_text(query, client)
    query_vector = query_embedding.reshape(1, -1)
    
    # FAISS 검색
    distances, indices = index.search(query_vector, top_k)
    
    # 결과 구성
    results = []
    for i, (dist, idx) in enumerate(zip(distances[0], indices[0])):
        if idx < len(metadata):
            result = {
                "rank": i + 1,
                "distance": float(dist),
                "chunk_id": metadata[idx]["chunk_id"],
                "content": metadata[idx]["content"]
            }
            results.append(result)
    
    return results

In [ ]:
# 검색 실행
query = "건축법의 목적은 무엇인가요?"
results = search_similar(query, index, metadata, client, top_k=3)

print(f"🔍 검색 쿼리: {query}")
print(f"\n검색 결과:")
for r in results:
    print(r)